In [12]:
# Import Required Libraries
import pandas as pd
from groq import Groq
import json
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Read GROQ API key from environment variable
api_key = os.getenv('GROQ_API_KEY')

# Check if API key exists
if api_key is None:
    raise ValueError("GROQ_API_KEY environment variable is not set. Please set it in your .env file.")

# Initialize Groq client with API key
client = Groq(api_key=api_key)

print("✓ All imports loaded successfully")
print("✓ GROQ API key loaded from environment")
print("✓ Groq client initialized")

✓ All imports loaded successfully
✓ GROQ API key loaded from environment
✓ Groq client initialized


In [13]:
import re

def extract_json_from_text(text):
    # Remove markdown code blocks if present
    text = text.strip()
    
    # Extract first JSON object using regex
    match = re.search(r'\{.*\}', text, re.DOTALL)
    if match:
        return match.group(0)
    
    return None

In [14]:
def detect_header_with_llm(file_path, sheet_name="Instant Orders", preview_rows=15):
    df = pd.read_excel(file_path, sheet_name=sheet_name, header=None, engine="openpyxl")
    preview_df = df.head(preview_rows)

    table_text = preview_df.to_string(index=True)

    prompt = f"""
You are a data analyst.

Below is a preview of an Excel sheet.

Identify which row index (0-based index shown on left) is the actual header row.

Rules:
- Header row contains column names.
- It is usually followed by actual data rows.
- It mostly contains strings.
- It is not a title or metadata row.

Return JSON only:
{{
    "header_row_index": number,
    "confidence": number_between_0_and_1,
    "reasoning": "short explanation"
}}

Excel Preview:
{table_text}
"""

    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )

    raw_content = response.choices[0].message.content.strip()
    json_text = extract_json_from_text(raw_content)

    if not json_text:
        raise ValueError("Model did not return valid JSON")

    return json.loads(json_text)

In [15]:
file_path = "/home/neosoft/Downloads/MehulAswar_0323202607130520260323-7-jpvg62.xlsx"

result = detect_header_with_llm(file_path, sheet_name="Instant Orders")
header_idx = result["header_row_index"]

print("Detected header row:", header_idx)

Detected header row: 8


In [16]:
df = pd.read_excel(
    file_path,
    sheet_name="Instant Orders",
    header=header_idx,
    engine="openpyxl"
)

df

,Trade ID,Crypto,Trade Completion time,Side (Buy/Sell),Avg Buying/Selling Price(in INR),Quantity,Gross Amount Paid/Received by the user(in INR),Fees(in INR),Net Amount Paid/Received by the user(in INR),*TDS(in INR)
0,e1eb73c8-0526-4c27-8bb2-6a488cc85a44,SOL,2026-03-20 14:08:24,Buy,8.539900e+03,2.328130e-02,198.82000,1.173038,199.993038,NaN
1,918e2a47-877c-461f-961f-5a1ef625040f,ETH,2026-03-17 14:51:11,Buy,2.172024e+05,1.373051e-03,298.23000,1.759557,299.989557,NaN
2,73578bf3-3262-40bd-9d6e-bf05fc6d95d4,BNB,2026-03-17 09:35:33,Buy,6.342821e+04,4.701851e-03,298.23000,1.759557,299.989557,NaN
3,320b1cdd-68fe-4581-a56c-b58db0ec9236,BTC,2026-03-17 07:17:32,Buy,7.075200e+06,4.215146e-05,298.23000,1.759557,299.989557,NaN
4,a31b0986-7fc2-43c1-9526-c800d004e4db,SOL,2026-03-13 15:23:18,Buy,8.355390e+03,2.379542e-02,198.82000,1.173038,199.993038,NaN
...,...,...,...,...,...,...,...,...,...,...
105,51d1129d-5915-43a0-ae7f-49623455fa24,PEPE,2025-05-18 20:39:19,Sell,1.187000e-03,1.384280e+06,1643.14036,9.694528,1633.445832,16.334458
106,f98d6cb5-31de-4b56-b169-6e910e1c883f,ETH,2025-05-14 01:31:20,Buy,2.262661e+05,4.393058e-03,994.00000,5.864600,999.864600,NaN
107,845f541d-2646-4a28-8254-77b0798dfe21,PEPE,2025-05-09 16:12:59,Buy,1.085000e-03,9.161290e+05,994.00000,5.864600,999.864600,NaN
108,640e2d35-8683-447f-b164-2550bdbd7f1f,ETH,2025-05-09 14:44:20,Buy,1.945192e+05,2.930302e-03,570.00000,3.363000,573.363000,NaN


In [17]:
def prepare_summary_input(df, max_rows=20):
    return df.head(max_rows).to_csv(index=False)

In [18]:
def summarize_trades_with_llm(csv_text):
    prompt = f"""
You are a financial analyst.

Below is a dataset of crypto trades.

Tasks:
1. Summarize total trades
2. Identify buy vs sell distribution
3. Highlight total investment (buy side)
4. Mention any quick insights (patterns, anomalies)

Keep it concise and structured.

Data:
{csv_text}
"""

    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )

    return response.choices[0].message.content.strip()

In [19]:
csv_text = prepare_summary_input(df)

summary = summarize_trades_with_llm(csv_text)

print(summary)

**Crypto Trade Summary (20 records – all Buy)**  

| Metric | Value |
|--------|-------|
| **Total trades** | **20** |
| **Buy vs Sell** | 20 Buy, 0 Sell (100 % buy side) |
| **Net amount paid (investment)** | **₹ 5,499.11** |
| **Total fees paid** | **₹ 32.25** (≈ 0.59 % of net spend) |
| **Average net spend per trade** | **≈ ₹ 274.96** |

---

### 1. Buy‑Side Investment Detail
- **SOL** – 5 purchases, total net outflow **₹ 999.97** (≈ 18 % of total).  
- **ETH** – 5 purchases, total net outflow **₹ 1,499.95** (≈ 27 %).  
- **BNB** – 5 purchases, total net outflow **₹ 1,499.95** (≈ 27 %).  
- **BTC** – 5 purchases, total net outflow **₹ 1,499.95** (≈ 27 %).  

*(Values rounded to two decimals.)*

---

### 2. Quick Insights & Patterns  

| Observation | Comment |
|-------------|---------|
| **All trades are buys** | No sell activity – likely a position‑building phase. |
| **Batch trading days** | 2026‑03‑03, 2026‑02‑24, 2026‑02‑17 each have three simultaneous trades (BTC, ETH, BNB). |
